In [1]:
%cd ..

/home/boat/proxyISP/pytorch-superpoint


/home/boat/miniconda3/envs/proxyopt/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import sys
import yaml
import torch
import pickle
import cv2
import rawpy
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "../ProxyOpt/")
sys.path.insert(0, "../fast-openISP/")
sys.path.insert(0, "../ProxyOpt/pytorch-msssim/")
from models.model_wrap import SuperPointFrontend_torch

# config_path = "/home/boat/proxyISP/pytorch-superpoint/configs/superpoint_coco_train_heatmap_proxyopt.yaml"
config_path = "/home/boat/proxyISP/pytorch-superpoint/configs/magicpoint_coco_export.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config.setdefault("model", {}).setdefault("subpixel", {})["enable"] = False

path = config["pretrained"]
assert "170000" in path
nms_dist = 4
nn_thresh = 0.7
conf_thresh = config["model"]["detection_threshold"]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

magicpoint_gpu_id = "1" if torch.cuda.device_count() > 1 else "0"
magicpoint_device = torch.device(f"cuda:{magicpoint_gpu_id}" if torch.cuda.is_available() else "cpu")
fe = SuperPointFrontend_torch(
        config=config,
        weights_path=path,
        nms_dist=nms_dist,
        conf_thresh=conf_thresh,
        nn_thresh=nn_thresh,
        cuda=False,
        device=magicpoint_device,
)

from ISP_tools.ProxyISPDataset import ProxyISPDataset, EXPERIMENT_OUTPUT_PATH

train_config_path = "/home/boat/proxyISP/ProxyOpt/train_configs/v16.2-chroma-HumanTunedInitialHype.yaml"

with open(train_config_path, "r") as f:
        yaml_dict = yaml.safe_load(f)

config = yaml_dict["config"]
openisp_config = yaml_dict["openisp_config"]
hyp_setting = yaml_dict["hyp_setting"]
additional_conf = {
    "proxyopt_base_path" : "/home/boat/proxyISP/ProxyOpt"
}
dataset = ProxyISPDataset(config, openisp_config, hyp_setting, additional_conf)

model:  SuperPointNet_gauss2


/home/boat/proxyISP/pytorch-superpoint/models/model_wrap.py:100: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weights_path,
[01-19 22:21:44] ProxyIS

self.config {'experiment_name': 'actual_proxy_v16.2-chroma', 'model_choice': 'proxyopt_unet', 'loss_choice': 'msssim', 'optimizer_choice': 'adam', 'input_type': 'stacked', 'output_domain': 'rgb', 'raw_path': '/mnt/ssd2tb/boat/thesis/s21fe_dataset', 'crop_mode': 'random', 'num_hyperp_to_sample': 50000, 'num_data_to_sample': False, 'img_size': 'dynamic', 'lr': 0.001, 'batch_size': 1, 'epoch': 200000, 'save_iter': 1000, 'save_only_latest': False, 'eval_iter': 10000, 'save_image_iter': 1000, 'lr_scheduler_iter': 13000, 'train_val_ratio': 0.85, 'gradient_accum_step': 1, 'memory_replay_prob': 0, 'memory_replay_buffer': 50, 'device': 'cuda', 'seed': 8888}


[01-19 22:21:45] ProxyISPDataset.py INFO - hyperp length: 50000
INFO:ISP_tools.ProxyISPDataset:hyperp length: 50000
[01-19 22:21:45] ProxyISPDataset.py INFO - num_hyperp x num_image: 53650000
INFO:ISP_tools.ProxyISPDataset:num_hyperp x num_image: 53650000
[01-19 22:21:53] ProxyISPDataset.py INFO - total dataset length: 53650000
INFO:ISP_tools.ProxyISPDataset:total dataset length: 53650000


In [3]:
import os
import glob
import pickle
import rawpy
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm import tqdm


# =========================
# Helper functions
# =========================

def keypoint_entropy_and_coverage(kps_xy, H, W, grid=(4, 4), eps=1e-12):
    """
    kps_xy: (N, 2) array of (x, y)
    H, W: image height and width
    Returns:
        entropy (normalized to [0,1])
        coverage ratio [0,1]
    """
    gx, gy = grid
    hist = np.zeros((gy, gx), dtype=np.float32)

    for x, y in kps_xy:
        ix = min(int(x / W * gx), gx - 1)
        iy = min(int(y / H * gy), gy - 1)
        hist[iy, ix] += 1

    # coverage
    coverage = (hist > 0).mean()

    # entropy
    p = hist.flatten()
    p = p / (p.sum() + eps)
    entropy = -np.sum(p * np.log(p + eps)) / np.log(len(p))

    return entropy, coverage


# =========================
# 1. USER CONFIG
# =========================

hype_paths = [
    "/home/boat/proxyISP/ProxyOpt/v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_adjust_defaultcolorhuesat/checkpoints/checkpoint_120000.pkl",
    "/home/boat/proxyISP/pytorch-superpoint/logs/PRETRAINED_SAMEMODELHOMOADAPT_AGGRESSIVEHOMOADAPT_CFANORMALIZE_XHOMOWARP_DETERMHOMOADAPT_train_v16.2-chroma-HumanTunedInitialHype_sunlit_lr0.005_bothLoss_initialHypeHomoAdaptOnly_gradac187/proxyopt_checkpoints/checkpoint_78500.pkl",
]

SEQ_PREFIX = "sl_*"

hpatches_root = "/mnt/ssd2tb/boat/thesis/s21fe_hpatches_v4"

RAW_CROP = (slice(540, 2460), slice(1040, 2960))
SP_SIZE = np.array([240, 320])  # (H, W)


# =========================
# 2. LOAD ALL HYPERPARAMS
# =========================

hypes = []
hype_names = []

for p in hype_paths:
    with open(p, "rb") as f:
        hyp = pickle.load(f)["proxy_hype"]
        hyp = dataset.denormalize_hyp(hyp)
    hypes.append(hyp)
    hype_names.append(os.path.basename(p))

# prepend original
hypes.insert(0, None)
hype_names.insert(0, "original")


# =========================
# 3. GLOB ALL DNG FILES
# =========================

dng_files = sorted(
    glob.glob(os.path.join(hpatches_root, SEQ_PREFIX, "*.dng"))
)

print(f"Found {len(dng_files)} DNG files")


# =========================
# 4. MAIN PIPELINE
# =========================

results = []

for idx, raw_path in enumerate(tqdm(dng_files, desc="Processing RAW images")):

    # ---- read raw ----
    with rawpy.imread(raw_path) as raw:
        bayer = raw.raw_image.copy()

    bayer = bayer[RAW_CROP]

    row = {
        "count": idx,
        "image": os.path.relpath(raw_path, hpatches_root),
    }

    for hyp, hyp_name in zip(hypes, hype_names):

        # ---- ISP processing ----
        if hyp is None:
            img = dataset.process_raw(bayer)
        else:
            img = dataset.process_raw(bayer, hyp)

        # ---- SuperPoint preprocessing ----
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        gray = cv2.resize(
            gray,
            (SP_SIZE[1], SP_SIZE[0]),
            interpolation=cv2.INTER_AREA
        )
        gray = gray.astype(np.float32) / 255.0
        gray = torch.from_numpy(gray)[None, None, :, :]  # [1,1,H,W]

        # ---- SuperPoint inference ----
        with torch.no_grad():
            pts, pts_desc, dense_desc, heatmap = fe.run(gray)

        # ---- metrics ----
        kp = pts[0]
        n_kp = kp.shape[1]

        if n_kp > 0:
            kps_xy = kp[:2].T  # (N, 2)
            entropy, coverage = keypoint_entropy_and_coverage(
                kps_xy,
                H=SP_SIZE[0],
                W=SP_SIZE[1],
                grid=(4, 4)
            )
        else:
            entropy, coverage = 0.0, 0.0

        row[f"{hyp_name}_count"] = n_kp
        row[f"{hyp_name}_entropy"] = entropy
        row[f"{hyp_name}_coverage"] = coverage

    results.append(row)


# =========================
# 5. BUILD PANDAS TABLE
# =========================

df = pd.DataFrame(results)
df = df.set_index("image")

print("\nKeypoint statistics table:")
print(df)


# =========================
# 6. MEAN & STD
# =========================

metric_cols = [
    c for c in df.columns
    if c.endswith(("_count", "_entropy", "_coverage"))
]

stats = pd.DataFrame({
    "mean": df[metric_cols].mean(),
    "std": df[metric_cols].std()
})

print("\nStatistics (per configuration & metric):")
print(stats)


/home/boat/miniconda3/envs/proxyopt/lib/python3.12/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


24 24
24 24
Found 102 DNG files


Processing RAW images:   0%|                  | 0/102 [00:00<?, ?it/s]

Executing awb... Done. Elapsed 0.019s
Executing cfa... Done. Elapsed 0.111s
Executing ccm... Done. Elapsed 0.108s
Executing gac... Done. Elapsed 0.048s
Executing bnfcv... Done. Elapsed 0.007s
Executing csc... Done. Elapsed 0.099s
Executing eeh... Done. Elapsed 0.150s
Executing hsc... Done. Elapsed 0.064s
Executing bcc... Done. Elapsed 0.019s
Pipeline elapsed 0.731s
Executing awb... Done. Elapsed 0.019s
Executing cfa... Done. Elapsed 0.113s
Executing ccm... Done. Elapsed 0.110s
Executing gac... Done. Elapsed 0.051s
Executing bnfcv... Done. Elapsed 0.038s
Executing csc... Done. Elapsed 0.134s
Executing eeh... Done. Elapsed 0.154s
Executing hsc... Done. Elapsed 0.063s
Executing bcc... Done. Elapsed 0.021s
Pipeline elapsed 0.858s
Executing awb... Done. Elapsed 0.012s
Executing cfa... Done. Elapsed 0.107s
Executing ccm... Done. Elapsed 0.090s
Executing gac... Done. Elapsed 0.049s
Executing bnfcv... Done. Elapsed 0.006s
Executing csc... Done. Elapsed 0.079s
Executing eeh... Done. Elapsed 0.1

Processing RAW images:   1%|          | 1/102 [00:02<04:39,  2.77s/it]

...

Executing awb... Done. Elapsed 0.012s
Executing cfa... Done. Elapsed 0.105s
Executing ccm... Done. Elapsed 0.113s
Executing gac... Done. Elapsed 0.051s
Executing bnfcv... Done. Elapsed 0.016s
Executing csc... Done. Elapsed 0.100s
Executing eeh... Done. Elapsed 0.155s
Executing hsc... Done. Elapsed 0.068s
Executing bcc... Done. Elapsed 0.019s
Pipeline elapsed 0.744s
Executing awb... Done. Elapsed 0.012s
Executing cfa... Done. Elapsed 0.104s
Executing ccm... Done. Elapsed 0.108s
Executing gac... Done. Elapsed 0.049s
Executing bnfcv... Done. Elapsed 0.031s
Executing csc... Done. Elapsed 0.102s
Executing eeh... Done. Elapsed 0.150s
Executing hsc... Done. Elapsed 0.064s
Executing bcc... Done. Elapsed 0.021s
Pipeline elapsed 0.746s
Executing awb... Done. Elapsed 0.012s
Executing cfa... Done. Elapsed 0.105s
Executing ccm... Done. Elapsed 0.110s
Executing gac... Done. Elapsed 0.049s
Executing bnfcv... Done. Elapsed 0.017s
Executing csc... Done. Elapsed 0.101s
Executing eeh... Done. Elapsed 0.1

Processing RAW images:  99%|███████▉| 101/102 [04:56<00:02,  2.86s/it]

Executing awb... Done. Elapsed 0.013s
Executing cfa... Done. Elapsed 0.114s
Executing ccm... Done. Elapsed 0.113s
Executing gac... Done. Elapsed 0.052s
Executing bnfcv... Done. Elapsed 0.017s
Executing csc... Done. Elapsed 0.100s
Executing eeh... Done. Elapsed 0.155s
Executing hsc... Done. Elapsed 0.066s
Executing bcc... Done. Elapsed 0.021s
Pipeline elapsed 0.756s
Executing awb... Done. Elapsed 0.014s
Executing cfa... Done. Elapsed 0.105s
Executing ccm... Done. Elapsed 0.088s
Executing gac... Done. Elapsed 0.046s
Executing bnfcv... Done. Elapsed 0.030s
Executing csc... Done. Elapsed 0.099s
Executing eeh... Done. Elapsed 0.157s
Executing hsc... Done. Elapsed 0.065s
Executing bcc... Done. Elapsed 0.022s
Pipeline elapsed 0.731s
Executing awb... Done. Elapsed 0.014s
Executing cfa... Done. Elapsed 0.105s
Executing ccm... Done. Elapsed 0.111s
Executing gac... Done. Elapsed 0.051s
Executing bnfcv... Done. Elapsed 0.018s
Executing csc... Done. Elapsed 0.102s
Executing eeh... Done. Elapsed 0.1

Processing RAW images: 100%|████████| 102/102 [04:59<00:00,  2.93s/it]


Keypoint statistics table:
                    count  original_count  original_entropy  \
image                                                         
sl_carforcat/1.dng      0             358          0.948620   
sl_carforcat/2.dng      1             355          0.952793   
sl_carforcat/3.dng      2             354          0.953003   
sl_carforcat/4.dng      3             368          0.957454   
sl_carforcat/5.dng      4             372          0.958199   
...                   ...             ...               ...   
sl_thaicult/2.dng      97             418          0.989477   
sl_thaicult/3.dng      98             391          0.984611   
sl_thaicult/4.dng      99             395          0.985977   
sl_thaicult/5.dng     100             422          0.983656   
sl_thaicult/6.dng     101             443          0.982865   

                    original_coverage  checkpoint_120000.pkl_count  \
image                                                                
sl_carforcat

In [4]:
df

,count,original_count,original_entropy,original_coverage,checkpoint_120000.pkl_count,checkpoint_120000.pkl_entropy,checkpoint_120000.pkl_coverage,checkpoint_78500.pkl_count,checkpoint_78500.pkl_entropy,checkpoint_78500.pkl_coverage
image,,,,,,,,,,
sl_carforcat/1.dng,0,358,0.948620,1.0,355,0.948052,1.0,355,0.948033,1.0
sl_carforcat/2.dng,1,355,0.952793,1.0,353,0.954317,1.0,352,0.953097,1.0
sl_carforcat/3.dng,2,354,0.953003,1.0,353,0.948623,1.0,357,0.951983,1.0
sl_carforcat/4.dng,3,368,0.957454,1.0,362,0.956753,1.0,362,0.951894,1.0
sl_carforcat/5.dng,4,372,0.958199,1.0,367,0.957559,1.0,371,0.958016,1.0
...,...,...,...,...,...,...,...,...,...,...
sl_thaicult/2.dng,97,418,0.989477,1.0,410,0.986852,1.0,418,0.989562,1.0
sl_thaicult/3.dng,98,391,0.984611,1.0,399,0.982435,1.0,391,0.984828,1.0
sl_thaicult/4.dng,99,395,0.985977,1.0,400,0.985752,1.0,398,0.985620,1.0


In [5]:
stats

,mean,std
original_count,391.882353,62.859626
original_entropy,0.970810,0.016278
original_coverage,1.000000,0.000000
checkpoint_120000.pkl_count,388.558824,62.474323
checkpoint_120000.pkl_entropy,0.969111,0.017813
checkpoint_120000.pkl_coverage,0.999387,0.006188
checkpoint_78500.pkl_count,390.225490,62.838297
checkpoint_78500.pkl_entropy,0.970440,0.016237
checkpoint_78500.pkl_coverage,0.999387,0.006188
